In [2]:
# A sample dataset to create a LLM (shakesphere dataset by andre karpathy)
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-27 19:40:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-07-27 19:40:00 (32.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
# reading it and converting it to text to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("the total length of all the charecters: ", len(text))


the total length of all the charecters:  1115394


In [5]:
#possible charecters the model can see or emit are usually created
# with a set and list ( to order and later to length it out)
char=sorted(list(set(text)))
vocablary_size= len(char)
print(''.join(char))
print(vocablary_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
#tokenize the text
#instead of the charecter level tokenizer we are implement sentence piece into the code
import sentencepiece as spm
#train
spm.SentencePieceTrainer.train(
    input="input.txt",
    model_prefix="tokenizer",
    vocab_size=5000,
    model_type="bpe"
)
#load
sp = spm.SentencePieceProcessor(model_file="tokenizer.model")

#creating fake stoi and itos (pieces)

stoi = {
    sp.id_to_piece(i): i
    for i in range(sp.get_piece_size())
}

itos = {
    i: sp.id_to_piece(i)
    for i in range(sp.get_piece_size())
}

def encode(s):
    return sp.encode(s, out_type=int)

def decode(ids):
    return sp.decode(ids)

#to test it like kar we are gonna use the exact same statements

print(encode("hiii there"))
print(decode(encode("hii there")))



[33, 4950, 4950, 4950, 255]
hii there


In [7]:
# tokenizing it entire set
#using pytorch
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([295231]) torch.int64
tensor([ 423,  807, 4964, 2088,   84, 2438,  548, 2014, 4956,  424,   68,  362,
        4965,  942, 4964, 2083, 4956,  362, 4965,  423,  807, 4964,  319,  182,
         157, 3722, 1234,   35,  712,  281,   35, 2523,  309, 4984,  942, 4964,
        3409,  250,  525, 4965, 3722, 4965,  423,  807, 4964,  423, 4956,   36,
         259, 3184, 1103,   76, 3464, 1682,   35,   13,  933, 4965,  942, 4964,
         393,  259, 4970, 4943, 4956,   84,  259, 4970, 4943, 4965,  423,  807,
        4964,  633,  285, 1058,  113, 4956,   45,   84, 4970,   21,  112, 2794,
         218,  178,  586,  834,   59, 4965,  438, 4970, 4943,    5, 3050, 4952,
        1528, 4984,  942, 4964,  439,  239, 1145,   47,  126, 4970, 4943, 4978,
         292,   91,   53,  589, 4964,  704, 4956,  704, 4986,  671,  807, 4964,
        1584,  457, 4956,  204, 2748, 4965,  423,  807, 4964,  393,  182, 2915,
          65,  701, 2748, 4956,   13, 4695,  204, 4965,  223, 3541, 2358,  274,
       

In [8]:
#seperate dataset

n=int(0.9*len(data))
train_data=data[:n]
val_data = data[n:]

In [9]:
# training dataset split into blocks called chuncks which is bascially a part of the dataset( a piece of it in simple terms)
block_size= 8
train_data[:block_size+1]
# as outpute would be the chunk of 9 charecters would be used in the transformer as x and y or the input and the target

tensor([ 423,  807, 4964, 2088,   84, 2438,  548, 2014, 4956])

In [10]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([423]) the target: 807
when input is tensor([423, 807]) the target: 4964
when input is tensor([ 423,  807, 4964]) the target: 2088
when input is tensor([ 423,  807, 4964, 2088]) the target: 84
when input is tensor([ 423,  807, 4964, 2088,   84]) the target: 2438
when input is tensor([ 423,  807, 4964, 2088,   84, 2438]) the target: 548
when input is tensor([ 423,  807, 4964, 2088,   84, 2438,  548]) the target: 2014
when input is tensor([ 423,  807, 4964, 2088,   84, 2438,  548, 2014]) the target: 4956


In [11]:
#seed the number seen is the same seen later
torch.manual_seed(1337)
batch_size=4
block_size=8#max context length

def get_batch(split):
  data= train_data if split=='train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[4964,  393, 4970,   21,  157, 3338,   36, 4978],
        [  91,  144,  307, 4956,   19,  144,   72,  108],
        [  91, 4965,  668, 4964,  319, 4984,  626, 4964],
        [2653,  456,   97,   80,  141, 2734, 4956,  206]])
targets:
torch.Size([4, 8])
tensor([[ 393, 4970,   21,  157, 3338,   36, 4978,   41],
        [ 144,  307, 4956,   19,  144,   72,  108, 2242],
        [4965,  668, 4964,  319, 4984,  626, 4964,  438],
        [ 456,   97,   80,  141, 2734, 4956,  206, 2092]])
----
when input is [4964] the target: 393
when input is [4964, 393] the target: 4970
when input is [4964, 393, 4970] the target: 21
when input is [4964, 393, 4970, 21] the target: 157
when input is [4964, 393, 4970, 21, 157] the target: 3338
when input is [4964, 393, 4970, 21, 157, 3338] the target: 36
when input is [4964, 393, 4970, 21, 157, 3338, 36] the target: 4978
when input is [4964, 393, 4970, 21, 157, 3338, 36, 4978] the target: 41
when input is [91] the target: 144


In [12]:
print(xb) # input to the transformer

tensor([[4964,  393, 4970,   21,  157, 3338,   36, 4978],
        [  91,  144,  307, 4956,   19,  144,   72,  108],
        [  91, 4965,  668, 4964,  319, 4984,  626, 4964],
        [2653,  456,   97,   80,  141, 2734, 4956,  206]])


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(sp.get_piece_size())
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 5000])
tensor(9.1076, grad_fn=<NllLossBackward0>)
 ⁇  suffic combeft enaid Sheys thought withalCA Becauseining liber senselish degdared broken intend demandkes understand some honoursaryENEN counteraies Boy hearts island dre Doth GREGORY iss duty Great dreamef leastkin civilvek wearursedAM alack deviseRIELESTER ado Anti cornULIET succouch untilRY afternoonowbever measure Beseechurp made money firesbES northakes Didst?-- QUEEN friendlyceive Hold ben conscience Christ lesser becomes purlt thrive silent deceitself no solemn hear JOHN vilevourTHUMantag Sic


In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32 # Increase batch size for training
block_size = 8 # Ensure block_size is consistent

# Number of training iterations
num_iterations = 5000

for iter in range(num_iterations):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0 or iter == num_iterations - 1:
        print(f"Iteration {iter}: Loss = {loss.item():.4f}")

Iteration 0: Loss = 8.9726
Iteration 500: Loss = 8.8299
Iteration 1000: Loss = 8.5877
Iteration 1500: Loss = 8.1343
Iteration 2000: Loss = 8.0123
Iteration 2500: Loss = 7.7597


KeyboardInterrupt: 

In [15]:
generated_text_indices = m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()
print(decode(generated_text_indices))

 ⁇  DERBY trulyrs action found est af slayellowrupt torlly Many great shame seestearuretyhestoke alter seeORIO!-- didst better food aught denol Prince ratherORD mile th army daughters pale resign flieslaw Pre hours French Bolingbroke ViennaISHOP will Pritheeentle Follow liege Dear Biond giftful ten ban speed DUKE manner found rich heaxumber impossible unkind g adversatswain pl noise repose North doing bonisper spour tarry soleated lettersUCH LUC Boling arriOMEO eagleanusomet pilgrim theirsaperldom see joy foul thing VIRG cry ten revengeaughters breath adver A tempest dreams woundennaaf pers quench Hunts POMP di mild cred commons merry business hear Of weddingound messility Hail overHOMASvere borne Even mistake spe list seas wallsning Conspirmber everyilling Yourso assuranceoy Tw sits Qantuabear silent suspect fri Sm Romceiun suppeth Mowb businessuries discover LAUR wat Froth inform my direct s maid once Provostirty cher no boot swallow buried shalt welcome yourselfclockisonval SLYvours

In [16]:
#self attention blocks (almost ready at this point)
#with the matrix multiplication trick
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [17]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [18]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [19]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [20]:
#matmul with weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [21]:
#softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [22]:
# self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)


head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [23]:
wei[0]


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [24]:
#scaling to control visualization
k= torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei= q@k.transpose(-2,-1)

In [25]:
torch.softmax(torch.tensor([0.1,0.2,0.3,-0.2,0.5])*8, dim= -1)

#softmax sharpening to the max

tensor([0.0305, 0.0678, 0.1510, 0.0028, 0.7479])

In [26]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    xmean = x.mean(1, keepdim=True)
    xvar = x.var(1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [27]:
# Feed Forward Network
# This gives each token a small neural network to process its information

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)

In [28]:
# Single self-attention head

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)

        wei = wei * head_size ** -0.5

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)

        v = self.value(x)

        out = wei @ v

        return out

In [29]:
# Multiple attention heads running in parallel

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        return out

In [30]:
# Transformer block
# Each block contains:
# LayerNorm -> Attention -> residual connection
# LayerNorm -> FeedForward -> residual connection

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

In [31]:
# GPT model configuration

batch_size = 32
block_size = 128

n_embd = 384
n_head = 6
n_layer = 6

dropout = 0.2

head_size = n_embd // n_head

print("Vocabulary size:", vocablary_size)
print("Embedding size:", n_embd)
print("Number of heads:", n_head)
print("Number of layers:", n_layer)
print("Block size:", block_size)

Vocabulary size: 65
Embedding size: 384
Number of heads: 6
Number of layers: 6
Block size: 128


In [32]:
# The complete GPT language model

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocablary_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd,
            vocablary_size
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:

            loss = None

        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(
                logits,
                dim=-1
            )

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [33]:
# Create the GPT model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

m = GPTLanguageModel().to(device)

print("Device:", device)

print(
    "Parameters:",
    sum(p.numel() for p in m.parameters()) / 1e6,
    "M"
)

Device: cuda
Parameters: 10.739777 M


In [34]:
# Get a batch of training data

def get_batch(split):

    data = train_data if split == 'train' else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [35]:
#SentencePiece vocabulary

vocablary_size = sp.get_piece_size()

print("SentencePiece vocabulary size:", vocablary_size)
print("Highest possible token ID:", vocablary_size - 1)

SentencePiece vocabulary size: 5000
Highest possible token ID: 4999


In [36]:
# Shakespeare dataset into SentencePiece token IDs

data = torch.tensor(
    sp.encode(text, out_type=int),
    dtype=torch.long
)

print("Total tokens:", len(data))
print("Minimum token ID:", data.min().item())
print("Maximum token ID:", data.max().item())

Total tokens: 295231
Minimum token ID: 0
Maximum token ID: 4998


In [37]:
# SentencePiece vocabulary size
vocablary_size = sp.get_piece_size()

print("Vocabulary size:", vocablary_size)
print("Minimum token ID:", min(data).item())
print("Maximum token ID:", max(data).item())

Vocabulary size: 5000
Minimum token ID: 0
Maximum token ID: 4998


In [38]:
# Recreation
m = BigramLanguageModel(vocablary_size)

print("Model vocabulary size:", m.token_embedding_table.num_embeddings)

Model vocabulary size: 5000


In [39]:
# Test

xb, yb = get_batch('train')

logits, loss = m(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

In [40]:
# Model configuration

batch_size = 32
block_size = 128

n_embd = 128
n_head = 4
n_layer = 4

dropout = 0.0

print("Vocabulary size:", vocablary_size)
print("Embedding dimension:", n_embd)
print("Attention heads:", n_head)
print("Transformer layers:", n_layer)

Vocabulary size: 5000
Embedding dimension: 128
Attention heads: 4
Transformer layers: 4


In [41]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)

        wei = wei * (k.shape[-1] ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(x)

        out = wei @ v

        return out

In [42]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(num_heads * head_size, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        out = self.dropout(out)

        return out

In [43]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [44]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

In [45]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocablary_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd,
            vocablary_size
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:

            loss = None

        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(
                logits,
                dim=-1
            )

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [46]:
torch.manual_seed(1337)

gpt = GPTLanguageModel()

print("GPT created successfully!")

print(
    "Number of parameters:",
    sum(p.numel() for p in gpt.parameters())
)

GPT created successfully!
Number of parameters: 2093192


In [47]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

gpt = gpt.to(device)

print("Using device:", device)

Using device: cuda


In [48]:
def get_batch_gpu(split):

    data_split = train_data if split == 'train' else val_data

    ix = torch.randint(
        len(data_split) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data_split[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data_split[i + 1:i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [49]:
xb, yb = get_batch_gpu('train')

logits, loss = gpt(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([32, 128])
Logits shape: torch.Size([4096, 5000])
Initial loss: 8.500986099243164


In [50]:
optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=3e-4
)

In [51]:
num_iterations = 5000

for iter in range(num_iterations):

    xb, yb = get_batch_gpu('train')

    logits, loss = gpt(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if iter % 500 == 0 or iter == num_iterations - 1:

        print(
            f"Iteration {iter}: "
            f"Loss = {loss.item():.4f}"
        )

Iteration 0: Loss = 8.4974
Iteration 500: Loss = 5.0480
Iteration 1000: Loss = 4.4311
Iteration 1500: Loss = 4.0122
Iteration 2000: Loss = 3.6081
Iteration 2500: Loss = 3.0580
Iteration 3000: Loss = 2.6362
Iteration 3500: Loss = 2.1601
Iteration 4000: Loss = 1.9444
Iteration 4500: Loss = 1.6425
Iteration 4999: Loss = 1.2681


In [52]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = gpt.generate(
    context,
    max_new_tokens=500
)

print(
    decode(generated[0].tolist())
)

 ⁇ EL: Hence, mount, and not thy lips here: there may be patient. In dust after-upid; sweet, I do ridget that, give thee beast; tell thy wretched man wrong? Fying by thy letters of thy life, That loss is banish'd: Flows, but far about me; Let me have age, and thou show for her no further. PRINCE: Come, cousin, women he may shed for you; and we will choose; but pity reconcile Thy legs Physic for what in'sty, Lest his thanks to Rome is the throats tales. SICINIUS: This know you: We cannot speak, perwack him since you his body caught up, Prove sweeter itself And meet you, rather stain the gates of my hazard. BRUTUS: With pates! VOLUMNIA: O, flowers, that, that would have rather I had My heart did kneel before what ever was mine own deeds dram! Then, that time thy manner doth me dem of my will import, But betide the subject of your grace When evil reveing answer, Think so much in many my care-proud Hast rich, nortled with thy royal execution cant than my blood, By such despair! Can practis

In [53]:
torch.save(
    gpt.state_dict(),
    "my_sentencepiece_gpt.pth"
)

print("Model saved!")

Model saved!
